# Cleanup workshop objects

Drops tables in the schema you pass, not every schema in the catalog. Confirm the widget values before you run.


## Bind the schema to tear down


In [ ]:
import sys
from pathlib import Path

dbutils.widgets.text("catalog", "YOUR_CATALOG", "Catalog")
dbutils.widgets.text("schema", "YOUR_SCHEMA", "Schema")
dbutils.widgets.text("warehouse_id", "YOUR_WAREHOUSE_ID", "SQL warehouse id")
dbutils.widgets.text("repo_root", "", "Workspace folder that contains shared/")

here = Path.cwd().resolve()
root = None
repo_root_w = dbutils.widgets.get("repo_root").strip()
if repo_root_w and (Path(repo_root_w) / "shared" / "config.py").exists():
    root = Path(repo_root_w)
if root is None:
    for candidate in [here, *here.parents]:
        if (candidate / "shared" / "config.py").exists():
            root = candidate
            break
if root is None:
    raise FileNotFoundError("shared/config.py not found. Set repo_root to the workspace folder that contains shared/.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from shared.config import load_config
from shared.lib.genie_client import GenieClient

cfg = load_config()
print("catalog", cfg.catalog)
print("schema", cfg.schema)
print("fqn", cfg.fqn)


## Drop tables in this schema only


In [ ]:
tables = [r["tableName"] for r in spark.sql(f"SHOW TABLES IN {cfg.fqn}").collect()]
print("tables", tables)
for name in tables:
    spark.sql(f"DROP TABLE IF EXISTS {cfg.fqn}.{name}")
    print("dropped table", name)
views = [r["viewName"] for r in spark.sql(f"SHOW VIEWS IN {cfg.fqn}").collect()] if False else []
try:
    views = [r["viewName"] for r in spark.sql(f"SHOW VIEWS IN {cfg.fqn}").collect()]
except Exception:
    views = []
for name in views:
    spark.sql(f"DROP VIEW IF EXISTS {cfg.fqn}.{name}")
    print("dropped view", name)
print("PASS schema emptied", cfg.fqn)
print("Genie agents are not deleted here. Delete them in the Genie UI if you created them for class.")
